In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
from skopt import gp_minimize
from skopt.space import Real, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt

# 1. 加载数据
print("正在加载数据...")
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 定义搜索空间 - 针对SVR的参数
dimensions = [
    Real(0.01, 10.0, "log-uniform", name='C'),
    Real(0.001, 0.1, "log-uniform", name='epsilon'),
    Real(0.01, 1.0, "log-uniform", name='gamma'),
    Categorical(['rbf', 'linear', 'poly'], name='kernel')
]

# 7. 定义目标函数 - 使用测试集R²作为优化目标
@use_named_args(dimensions)
def objective(C, epsilon, gamma, kernel):
    # 组合参数
    params = {
        'C': C,
        'epsilon': epsilon,
        'gamma': gamma,
        'kernel': kernel,
        'cache_size': 1000  # 缓存大小，加速训练
    }
    
    # 使用参数创建模型
    base_model = SVR(**params)
    model = MultiOutputRegressor(base_model)
    
    try:
        # 在训练集上训练模型
        model.fit(X_train_scaled, y_train)
        
        # 在测试集上预测
        y_pred = model.predict(X_test_scaled)
        
        # 计算测试集上每个目标变量的R²
        r2_scores = []
        for idx in range(len(target_vars)):
            r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
            r2_scores.append(r2)
        
        # 计算平均R²并取负值（因为优化器是最小化函数）
        mean_r2 = np.mean(r2_scores)
        
        # 记录当前迭代的参数和R²
        print(f"参数: C={C:.5f}, epsilon={epsilon:.5f}, gamma={gamma:.5f}, kernel={kernel} - 平均R²: {mean_r2:.4f}")
        
        return -mean_r2  # 返回负R²，因为我们要最大化R²，而优化器是最小化函数
    
    except Exception as e:
        print(f"错误: {e}")
        return 0  # 出错时返回一个较差的值

# 8. 执行贝叶斯优化
print("\n开始基于测试集R²的SVR贝叶斯参数优化...")
result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,  # 迭代次数
    n_random_starts=10,  # 随机搜索起点数
    random_state=42,
    verbose=True,
    n_jobs=-1
)

# 10. 提取最佳参数
best_params = {
    'C': result.x[0],
    'epsilon': result.x[1],
    'gamma': result.x[2],
    'kernel': result.x[3],
    'cache_size': 1000
}

print("\n最优参数:")
for param, value in best_params.items():
    if param != 'cache_size':
        print(f"{param}: {value}")

# 11. 使用最优参数创建新模型
print("\n用最优参数训练最终模型...")
best_model = MultiOutputRegressor(SVR(**best_params))

# 12. 用原始训练数据训练最终模型
best_model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 13. 在测试集上进行预测
y_pred = best_model.predict(X_test_scaled)

# 14. 评估最终模型性能
print("\n优化后模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 15. 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 16. 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳R²: {-result.fun:.4f}")  # 去掉负号，因为我们优化时是负R²
print(f"优化迭代次数: {len(result.func_vals)}")

# 17. 为比较创建一个基准模型（使用SVR.ipynb中的参数作为基准）
print("\n训练对照模型...")
baseline_params = {
    'kernel': 'rbf',
    'C': 0.11552292675414941,
    'epsilon': 0.01,
    'gamma': 0.26208814362264377,
    'cache_size': 1000
}

baseline_model = MultiOutputRegressor(SVR(**baseline_params))

# 用原始训练数据训练基准模型
baseline_model.fit(X_train_scaled, y_train)

# 在测试集上评估基准模型
y_pred_baseline = baseline_model.predict(X_test_scaled)

# 计算基准模型指标
baseline_r2 = 0
baseline_rmse = 0
for idx in range(len(target_vars)):
    r2 = r2_score(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    rmse = np.sqrt(mse)
    baseline_r2 += r2
    baseline_rmse += rmse

baseline_r2 /= len(target_vars)
baseline_rmse /= len(target_vars)

print(f"\n基准模型平均R²: {baseline_r2:.4f}")
print(f"基准模型平均RMSE: {baseline_rmse:.4f}")
print(f"优化模型平均R²: {overall_r2:.4f}")
print(f"优化模型平均RMSE: {overall_rmse:.4f}")
print(f"R²改进: {(overall_r2 - baseline_r2):.4f}")
print(f"RMSE改进: {(baseline_rmse - overall_rmse):.4f}")

# 可选：绘制优化过程
plt.figure(figsize=(10, 6))
plt.plot(-np.array(result.func_vals))
plt.xlabel('迭代次数')
plt.ylabel('平均R²')
plt.title('SVR贝叶斯优化过程')
plt.grid(True)
plt.show()

# 保存最优参数到文件
pd.DataFrame({
    'parameter': list(best_params.keys()),
    'value': list(best_params.values())
}).to_csv('svr_best_params.csv', index=False)